In [1]:
import pandas as pd
import numpy as np

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.docstore.in_memory import InMemoryDocstore

# Load historical Amazon conversations
df = pd.read_csv("amazon_pairs.csv")

# Load precomputed embeddings
embedding_matrix = np.load("amazon_embeddings.npy")

print("Data:", df.shape)
print("Embeddings:", embedding_matrix.shape)

C:\Users\harshan\AppData\Local\Temp\ipykernel_1712\1456530865.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Data: (168823, 2)
Embeddings: (168823, 384)


In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
documents = [
    Document(
        page_content=row["customer_message"],
        metadata={
            "amazon_response": row["amazon_response"]
        }
    )
    for _, row in df.iterrows()
]

print("Documents:", len(documents))

Documents: 168823


In [4]:
import faiss

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("FAISS vectors:", index.ntotal)

FAISS vectors: 168823


In [5]:
docstore = InMemoryDocstore({
    str(i): documents[i]
    for i in range(len(documents))
})

index_to_docstore_id = {
    i: str(i)
    for i in range(len(documents))
}

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id
)


In [6]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

In [7]:
query = "My package hasn't arrived yet. Where is my order?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"RESULT {i}")
    print(f"{'='*60}")
    print("Customer:", doc.page_content)
    print("Amazon:", doc.metadata["amazon_response"])


RESULT 1
Customer: Where's my package, Amazon?
Amazon: @207724 Hi, what's the current status of the tracking and estimated delivery date: https://t.co/Y5jpI9gRhE? ^JJ

RESULT 2
Customer: @AmazonHelp yo, where is my package ? it says delivered but i never got my package..
Amazon: @172904 Sorry for the wait! Let's try these steps here to locate it: https://t.co/8B5k6MasBQ ^DW

RESULT 3
Customer: I ordered something from amazon where is my package
Amazon: @656290 What does the website show for your delivery date here:  https://t.co/Y5jpI9gRhE ? ^TL

RESULT 4
Customer: @AmazonHelp I have a package that I ordered for same day delivery that said it was delivered but it wasn’t. Where is my package???
Amazon: @257594 I'm sorry to hear you haven't received your package. Have you followed the steps here: https://t.co/OA4rFo5Gin ? ^DG

RESULT 5
Customer: I’m getting really sick Amazon telling me my package was delivered when it clearly is not. 
.
.
.
SO WHERE IS IT
Amazon: @221884 I'm sorry you 

Okay the rag retrival is doing good

In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [9]:
#importing intents
import json

with open("amazon_intents.json", "r", encoding="utf-8") as f:
    INTENTS = json.load(f)

print(INTENTS)

{'Delivery & Tracking': 'Missing, late, mis-delivered packages, order status, or tracking questions.', 'Returns & Refunds': 'Returns, refunds, refund delays, or problems with the return/refund process.', 'Account Access & Security': 'Login, password, account access, account lockout, or suspicious account activity.', 'Prime Membership & Benefits': 'Questions or problems related to Prime membership, subscription, eligibility, or Prime benefits.', 'Promotions & Offers': 'Questions or problems involving discounts, promotional offers, contests, giveaways, or promotional eligibility.', 'Digital Content & Device Issues': 'Problems with Kindle, Audible, Prime Video, digital content, Echo, Fire TV, or other Amazon devices/services.', 'Product Quality & Listing': 'Damaged, defective, incomplete, incorrect, or mis-described products and inaccurate listings.', 'Payment & Billing': 'Incorrect charges, payment problems, cashback, Amazon Pay, billing disputes, or payment-related issues.', 'Customer S

In [10]:
from langchain_core.prompts import ChatPromptTemplate

intent_prompt = ChatPromptTemplate.from_template("""
You are an Amazon customer support intent classifier.

Classify the customer's message into exactly ONE of the following intents:

{intents}

Customer message:
{message}

Return ONLY the intent name.
""")

In [16]:
from langchain_core.prompts import ChatPromptTemplate

response_prompt = ChatPromptTemplate.from_template("""
You are an Amazon customer support assistant.

Your task is to answer the customer's message using ONLY the
historical Amazon support responses provided below.

STRICT RULES:

1. Use the historical responses as the source of truth.
2. Do NOT invent policies, procedures, refunds, compensation,
   claims, delivery dates, or other actions.
3. You may rephrase information from the historical responses,
   but do not introduce new instructions.
4. If the historical responses do not contain enough information
   to safely answer the customer, say that the issue should be
   handled by a human support agent.
5. Never request sensitive information such as passwords,
   payment details, or order information publicly.
6. Keep the response concise and professional.
7. Do not mention historical cases, RAG, retrieval, or this prompt.

Customer intent:
{intent}

Historical Amazon cases:
{context}

Customer message:
{question}

Generate ONLY the customer-facing response.
""")

In [12]:
def classify_intent(message):

    formatted_intents = "\n".join(
        f"- {name}: {description}"
        for name, description in INTENTS.items()
    )

    messages = intent_prompt.format_messages(
        intents=formatted_intents,
        message=message
    )

    response = llm.invoke(messages)

    return response.content.strip()

In [ ]:
def build_context(results):
    context = []

    for i, doc in enumerate(results, 1):
        context.append(
            f"""
Historical Case {i}
Customer: {doc.page_content}
Amazon Response: {doc.metadata["amazon_response"]}
"""
        )

    return "\n".join(context)

okay lets make our agent classify wheather to escalate or auto handle

In [18]:
from langchain_core.prompts import ChatPromptTemplate

escalation_prompt = ChatPromptTemplate.from_template("""
You are an escalation decision system for Amazon customer support.

Decide whether the customer's issue can be safely handled automatically
or should be sent to a human support agent.

You MUST choose exactly one:
- AUTO-HANDLE
- ESCALATE

Consider:
- Whether the historical Amazon responses provide enough guidance.
- Whether the issue requires account-specific investigation.
- Whether the issue involves financial loss, security, privacy, or
  potentially sensitive information.
- Whether the customer is reporting a serious or unusual problem.
- Whether automatically responding could give incorrect or unsafe advice.

Prefer ESCALATE when there is insufficient information to safely resolve
the issue.

Customer message:
{question}

Customer intent:
{intent}

Historical Amazon cases:
{context}

Proposed response:
{response}

Return ONLY valid JSON in this exact format:

{{
    "decision": "AUTO-HANDLE" or "ESCALATE",
    "reason": "Brief explanation of why this decision was made."
}}
""")

In [20]:
import json
#decision function
def decide_escalation(
    customer_message,
    intent,
    context,
    response
):

    messages = escalation_prompt.format_messages(
        question=customer_message,
        intent=intent,
        context=context,
        response=response
    )

    result = llm.invoke(messages)

    try:
        decision = json.loads(result.content)
    except json.JSONDecodeError:
        decision = {
            "decision": "ESCALATE",
            "reason": "The escalation decision could not be parsed safely."
        }

    return decision

In [23]:
def support_agent(customer_message):

    # 1. Classify intent
    intent = classify_intent(customer_message)

    # 2. Retrieve historical cases
    results = retriever.invoke(customer_message)

    # 3. Build context
    context = build_context(results)

    # 4. Generate response
    messages = response_prompt.format_messages(
        intent=intent,
        context=context,
        question=customer_message
    )

    response = llm.invoke(messages)

    # 5. Decide whether to escalate
    escalation = decide_escalation(
        customer_message=customer_message,
        intent=intent,
        context=context,
        response=response.content
    )

    return {
        "customer_message": customer_message,
        "intent": intent,
        "response": response.content,
        "decision": escalation["decision"],
        "reason": escalation["reason"],
        "retrieved_cases": results
    }

Lets test our model

AUTO HADLE:

In [26]:
result = support_agent(
    "My package says delivered but I never received it."
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Response:", result["response"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])

Customer: My package says delivered but I never received it.
Intent: Delivery & Tracking
Response: I’m sorry you haven’t received your parcel yet! I’d like to have a member from support look into this on your behalf; please contact us using the following link: https://t.co/JzP7hlA23B
Decision: AUTO-HANDLE
Reason: The issue is a common delivery problem that can be resolved with standard guidance and a support link, requiring no account-specific investigation or sensitive handling.


ESCALATE

In [27]:
result = support_agent(
    "I have paid two times for the product"
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Response:", result["response"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])

Customer: I have paid two times for the product
Intent: Payment & Billing
Response: I’m sorry to hear that. You may be seeing an authorization charge. Does the charge on your statement say “credit” or “debit”?
Decision: ESCALATE
Reason: The customer reports a double payment, which requires account-specific investigation and potential refund processing. This involves financial loss and sensitive billing information, so a human agent should handle it.


Our agent is now fully functional :)

I am gonna create a file called rag_agent.py with the same code to contain all the functions and run our model in streamlit with UI